#### T03 Layers Customization

Route traffic by **layer name** (`api` vs `worker`): different handlers, different files, same logger — mirrors real services.

`examples/config/enterprise_multi_layer_api_worker.yaml`

**Expect:** Layers `api` / `worker` → `api.jsonl` and `worker.jsonl` under `examples/logs/notebooks/`. <small>More: `examples/tutorials/notebooks/README.md`.</small>


**§0 — pip**  
<small>Optional if `import hydra_logger` already works.</small>


In [1]:
%pip install -q "hydra-logger"
# Restart kernel if pip upgraded deps.


Note: you may need to restart the kernel to use updated packages.


**§1 — Setup**  
<small>Notebook plumbing (usually **collapsed**): ``importlib`` loads ``jupyter_workspace.py``, then **one call** — ``prime_notebook_workspace()`` — (``shared.path_bootstrap`` → ``utility``). Clone root follows the loaded file, not Jupyter's cwd. Presets: ``examples/config/``. Details: `notebooks/README.md`.</small>


In [2]:
import importlib.util
import os
import tempfile
from pathlib import Path

def _resolved_notebook_cwd() -> Path:
    try:
        return Path.cwd().resolve()
    except FileNotFoundError:
        fb = Path(tempfile.gettempdir()).resolve()
        os.chdir(fb)
        return fb

def _load_jupyter_workspace_module():
    _starts: list[Path] = []
    _env = os.environ.get("HYDRA_LOGGER_REPO", "").strip()
    if _env:
        _starts.append(Path(_env).expanduser().resolve())
    _starts.append(_resolved_notebook_cwd())
    _seen: set[str] = set()
    for _s in _starts:
        for _b in (_s, *_s.parents):
            _jw = _b / "examples" / "tutorials" / "jupyter_workspace.py"
            try:
                _key = str(_jw.resolve())
            except (OSError, RuntimeError):
                continue
            if _key in _seen:
                continue
            _seen.add(_key)
            if _jw.is_file():
                _spec = importlib.util.spec_from_file_location("_hydra_tutorial_jw", _jw)
                assert _spec and _spec.loader
                _mod = importlib.util.module_from_spec(_spec)
                _spec.loader.exec_module(_mod)
                return _mod
    raise FileNotFoundError(
        "Could not find examples/tutorials/jupyter_workspace.py. Clone hydra-logger, set "
        "HYDRA_LOGGER_REPO to the clone (or any directory inside it), or start Jupyter with cwd "
        "inside that clone."
    )

repo_root = _load_jupyter_workspace_module().prime_notebook_workspace()


Repo root (cwd): /home/razvansavin/Projects/hydra-logger


**§2 — Imports**

In [3]:
from hydra_logger import create_sync_logger


**§3 — Config path**  
<small>`CONFIG_PATH` is ``repo_root / "examples" / "config" / …`` — the same files you edit in the repository.</small>


In [4]:
CONFIG_PATH = repo_root / "examples" / "config" / "enterprise_multi_layer_api_worker.yaml"


**§3b — JSON preset on disk**  
<small>`LoggingConfig` is the same whether you load YAML (§5) or validate JSON. Here we peek at `dev_console_file.json` — §5 still uses the YAML `CONFIG_PATH`.</small>


In [5]:
import json

from hydra_logger.config.models import LoggingConfig

_JSON_PRESET = repo_root / "examples" / "config" / "dev_console_file.json"
_raw = json.loads(_JSON_PRESET.read_text(encoding="utf-8"))
_from_json = LoggingConfig.model_validate(_raw)
print("JSON preset:", _JSON_PRESET.name, "→ layers:", list(_from_json.layers.keys()))


JSON preset: dev_console_file.json → layers: ['app']


**§5 — Scenario**  
<small>Your hydra-logger usage pattern.</small>


In [6]:
with create_sync_logger(
    config_path=CONFIG_PATH,
    strict_unknown_fields=True,
    name="tutorial-t03",
) as logger:
    logger.info("request ok", layer="api", context={"route": "/v1/orders"})
    logger.info("job running", layer="worker", context={"job_id": "job-1"})


| 2026-03-21 17:18:14 | INFO | api | request ok


**§6 — Results**  
<small>Log file tails under ``examples/logs/notebooks/``.</small>


In [7]:
_root = (repo_root / "examples" / "logs" / "notebooks")
print("Last lines (if files exist):")
for _name in ("api.jsonl", "worker.jsonl"):
    _p = _root / _name
    print(f"--- {_name} ---")
    if _p.is_file():
        for _line in _p.read_text(encoding="utf-8").splitlines()[-6:]:
            print(_line)


Last lines (if files exist):
--- api.jsonl ---
{"timestamp":"2026-03-21 16:52:37","level":20,"level_name":"INFO","message":"request ok","logger_name":"tutorial-t03","layer":"api","file_name":"1041656408.py","function_name":"<module>","line_number":6}
{"timestamp":"2026-03-21 17:18:14","level":20,"level_name":"INFO","message":"request ok","logger_name":"tutorial-t03","layer":"api","file_name":"1041656408.py","function_name":"<module>","line_number":6}
--- worker.jsonl ---
{"timestamp":"2026-03-21 16:52:37","level":20,"level_name":"INFO","message":"job running","logger_name":"tutorial-t03","layer":"worker","file_name":"1041656408.py","function_name":"<module>","line_number":7}
{"timestamp":"2026-03-21 17:18:14","level":20,"level_name":"INFO","message":"job running","logger_name":"tutorial-t03","layer":"worker","file_name":"1041656408.py","function_name":"<module>","line_number":7}


**Iterate:** edit YAML under `examples/config/`, re-run **§5** (and **§6** if present). <small>Details: `notebooks/README.md`.</small>
